# 03. curves: 레시피 + 시세 → QuantLib 커브, 그리고 흔들 수 있는 커브

`src/quanttool/market/curves.py`

| 의존하는 것 | 제공하는 것 |
|---|---|
| `config` (관례·커브 레시피·지수), 시세 하루치 | `build_curve` → `BuiltCurve`, `HELPERS`, 진단 표, `CurveFactors`(시장 객체) |

**객체 지도** (이 순서로 본다)

1. 날짜 도구: `to_ql_date`, `evaluation_date`, `bill_price`/`bill_yield`
2. 결과 묶음: `Context`, `Helper`, `BuiltCurve`
3. `HELPERS`: 구간 종류 → 헬퍼 함수 (5종 하나씩)
4. `build_curve`: 얼림 vs linked, QuantLib 직접 조립과 비교, 듀얼 커브
5. 진단: `helper_repricing`, `curve_table`, `curve_shape`, `conventions_for`
6. `CurveFactors`: 기준 커브 + 격자 이동량 층, `roll`

> 데이터는 `tests/fixtures/`의 **합성 샘플**이다. 구조를 보는 용도이고 숫자로 시장을 해석하지 않는다.

In [ ]:
from datetime import date
from pathlib import Path

import pandas as pd
import QuantLib as ql

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
FIX = ROOT / "tests" / "fixtures"  # 합성 샘플 시세·예시 북
ASOF = date(2026, 10, 23)

from quanttool.config import load_config
from quanttool.market.quotes import read_quotes

cfg = load_config(ROOT / "configs")
quotes = read_quotes(FIX / "market_sample.txt")
q = quotes[ASOF]

## 날짜 도구
`evaluation_date(d)`는 QuantLib 전역 평가일을 잠시 d로 두고 블록이 끝나면 되돌린다.

In [ ]:
from quanttool.market.curves import bill_price, bill_yield, evaluation_date, to_ql_date

S = ql.Settings.instance()
print("to_ql_date:", to_ql_date(ASOF))
before = S.evaluationDate
with evaluation_date(ASOF):
    print("블록 안:", S.evaluationDate)
print("블록 밖 되돌림:", S.evaluationDate == before)
p = bill_price(0.0395, 91)
print("bill BEY 3.95%, 91일 → 가격", round(p, 6), "→ 다시 수익률", round(bill_yield(p, 91), 10))

## 결과 묶음 (dataclass)
- `Context`: 헬퍼 함수가 받는 재료 (커브 이름, 관례, 달력, 결제일, 할인 Handle, 지수 만드는 함수)
- `Helper`: 헬퍼 하나 (시세 키, 시세 Quote, QuantLib 헬퍼, 만기, 헬퍼 단위 → 시세 단위 함수)
- `BuiltCurve`: 커브 하나 (이름, 부트스트랩 커브, 헬퍼들)

In [ ]:
from dataclasses import fields

from quanttool.market.curves import BuiltCurve, Context, Helper

{c.__name__: [f.name for f in fields(c)] for c in [Context, Helper, BuiltCurve]}

## `HELPERS`: 구간 종류 → 헬퍼 함수
같은 "커브"라도 시세 종류에 따라 헬퍼가 다르다. 함수 모양은 모두 `(키, Quote, 만기, 구간, ctx) → Helper`.

In [ ]:
from quanttool.market.curves import HELPERS


def ctx_for(curve, discount=None):
    cc, conv = cfg.curves[curve], cfg.conventions[cfg.curves[curve].conventions]
    cal, today = conv.ql_calendar(), to_ql_date(ASOF)
    return cc, Context(
        curve,
        conv,
        cal,
        cal.advance(today, conv.settlement_days, ql.Days),
        discount or ql.YieldTermStructureHandle(),
        lambda n: cfg.indices[n].make(n),
    )


rows = []
with evaluation_date(ASOF):
    for curve, seg_i, tenor in [
        ("UST", 0, "3M"),
        ("UST", 1, "10Y"),
        ("KOFR", 0, "1Y"),
        ("CD91", 0, "3M"),
        ("CD91", 1, "5Y"),
    ]:
        cc, ctx = ctx_for(curve)
        seg = cc.segments[seg_i]
        key = cc.key(seg, tenor)
        h = HELPERS[seg.type](key, ql.SimpleQuote(q[key]), tenor, seg, ctx)
        rows.append(
            {
                "구간 종류": seg.type,
                "key": h.key,
                "시세": h.quote.value(),
                "QuantLib 헬퍼": type(h.helper).__name__,
                "헬퍼가 맞추는 값": h.helper.quote().value(),
                "만기": h.maturity.ISO(),
            }
        )
pd.DataFrame(rows)

Bill·Bond 헬퍼는 **가격**(할인채 가격, par 채권 100)을 맞추고, OIS·Deposit·Swap은 **금리**를 맞춘다. `Helper.to_quote`가 가격을 시세 단위(수익률)로 되돌린다.

## `build_curve(레시피, 관례, 시세, asof, linked, curves, indices)` → `BuiltCurve`

In [ ]:
from quanttool.market.curves import build_curve

ust = build_curve(cfg.curves["UST"], cfg.conventions["UST"], q, ASOF)
print(
    ust.name, type(ust.curve).__name__, len(ust.helpers), "헬퍼, 기준일", ust.curve.referenceDate()
)
list(ust.quotes())[:4]

QuantLib으로 직접 조립한 커브와 같다 (bill: BEY → 할인채 가격, 쿠폰채: par yield = 쿠폰인 가격 100 채권).

In [ ]:
with evaluation_date(ASOF):
    today = to_ql_date(ASOF)
    cal = ql.UnitedStates(ql.UnitedStates.GovernmentBond)
    settle = cal.advance(today, 1, ql.Days)
    hs = []
    for t in ["1M", "2M", "3M", "4M", "6M", "1Y"]:
        mat = cal.adjust(settle + ql.Period(t), ql.Following)
        zcb = ql.ZeroCouponBond(1, cal, 100.0, mat, ql.Following, 100.0, settle)
        price = bill_price(q[f"BOND/YIELD/USD/UST/PAR/{t}"], mat - settle)
        hs.append(ql.BondHelper(ql.QuoteHandle(ql.SimpleQuote(price)), zcb))
    for t in ["2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]:
        sch = ql.Schedule(
            settle,
            settle + ql.Period(t),
            ql.Period(ql.Semiannual),
            cal,
            ql.Unadjusted,
            ql.Unadjusted,
            ql.DateGeneration.Backward,
            False,
        )
        hs.append(
            ql.FixedRateBondHelper(
                ql.QuoteHandle(ql.SimpleQuote(100.0)),
                1,
                100.0,
                sch,
                [q[f"BOND/YIELD/USD/UST/PAR/{t}"]],
                ql.ActualActual(ql.ActualActual.Bond),
                ql.Unadjusted,
                100.0,
                settle,
            )
        )
    direct = ql.PiecewiseLogCubicDiscount(today, hs, ql.Actual365Fixed())
    print("10Y 할인계수 차이:", ust.curve.discount(10.0) - direct.discount(10.0))

**얼림 vs linked**: 기본은 부트스트랩 직후 얼려 시세가 바뀌어도 그대로다. `linked=True`면 시세 Quote를 바꾸면 커브가 다시 풀린다.

In [ ]:
for linked in [False, True]:
    b = build_curve(cfg.curves["UST"], cfg.conventions["UST"], q, ASOF, linked=linked)
    s = b.quotes()["BOND/YIELD/USD/UST/PAR/10Y"]
    with evaluation_date(ASOF):
        before = b.curve.zeroRate(10.0, ql.Continuous).rate()
        s.setValue(s.value() + 0.0010)
        print(
            f"linked={linked}: 10Y 시세 +10bp → 10Y 제로 {(b.curve.zeroRate(10.0, ql.Continuous).rate() - before) * 1e4:.2f}bp"
        )

**듀얼 커브**: CD91의 스왑 헬퍼는 `curves`로 받은 KOFR Handle로 할인한다(`discount_curve: KOFR`). 그래서 KOFR를 먼저 만든다.

In [ ]:
kofr = build_curve(cfg.curves["KOFR"], cfg.conventions["KRW_OIS"], q, ASOF, indices=cfg.indices)
handles = {"KOFR": ql.YieldTermStructureHandle(kofr.curve)}
cd91 = build_curve(
    cfg.curves["CD91"], cfg.conventions["KRW_IRS"], q, ASOF, curves=handles, indices=cfg.indices
)
with evaluation_date(ASOF):
    print(
        {b.name: round(b.curve.zeroRate(5.0, ql.Continuous).rate() * 100, 4) for b in [kofr, cd91]},
        "(5Y 제로 %)",
    )

## 진단
- `helper_repricing`: 커브로 다시 계산한 시세 − 시장 시세(bp). 0에 가까워야 믿는다.
- `curve_table`: 만기별 할인계수, 제로(연속·반기), 1Y 선도, par
- `curve_shape`: 10Y 수준, 2s10s, 5s30s, 나비 (bp)
- `conventions_for(cfg, 커브)`: 그 커브의 관례

In [ ]:
from quanttool.market.curves import conventions_for, curve_shape, curve_table, helper_repricing

display(helper_repricing(cd91, ASOF))
with evaluation_date(ASOF):
    table = curve_table(ql.YieldTermStructureHandle(ust.curve), conventions_for(cfg, "UST"))
display((table[["discount", "zero_cc", "fwd_1y", "par"]]).round(5))
curve_shape(table)

## `CurveFactors`: 기준 커브 + 격자 이동량 층 (시장 객체)
```
handle (상품이 연결) → PiecewiseZeroSpreadedTermStructure(_base_link, 격자 이동량 Quote들)
                                                  └ _base_link → 기준 커브 (roll이 갈아 끼움)
```
이동량이 0이면 기준 커브와 정확히 같다. 격자점 사이는 선형, 바깥은 평탄.

In [ ]:
from quanttool.market.curves import CurveFactors

cf = CurveFactors(ust.curve, ASOF, cfg.curves["UST"].factor_tenors)
print("tenors", cf.tenors[:6], "…")
print("times", {k: round(v, 4) for k, v in list(cf.times.items())[:4]}, "…")
print(
    "factors",
    type(cf.factors()["5Y"]).__name__,
    cf.factors()["5Y"].value(),
    "| handle",
    type(cf.handle).__name__,
)
with evaluation_date(ASOF):
    print("기준 상태 차이:", cf.handle.discount(7.3) - cf.base.discount(7.3))

5Y 이동량 하나를 1bp로: 5Y에서 1bp, 4.5Y(4Y·5Y 가운데)에서 0.5bp, 이웃 격자점 4Y·6Y에서 0.

In [ ]:
def zero_bp(c, t):
    return c.zeroRate(t, ql.Continuous).rate() * 1e4


t = cf.times
cf.factors()["5Y"].setValue(1e-4)
with evaluation_date(ASOF):
    at = {"4Y": t["4Y"], "4.5Y": (t["4Y"] + t["5Y"]) / 2, "5Y": t["5Y"], "6Y": t["6Y"]}
    moved = pd.Series(
        {k: zero_bp(cf.handle, x) - zero_bp(cf.base, x) for k, x in at.items()}, name="제로 이동 bp"
    )
cf.factors()["5Y"].setValue(0.0)
moved.round(4)

`roll(d)`: 기준 커브를 `ImpliedTermStructure(오늘 커브, d)`로 갈아 끼운다 (ForwardForward: DF_d(t) = DF(d+t)/DF(d)). `refresh()`는 할 일이 없다(Observer).

In [ ]:
d = date(2026, 10, 26)
cf.roll(d)
with evaluation_date(d):
    implied = ql.ImpliedTermStructure(ql.YieldTermStructureHandle(ust.curve), to_ql_date(d))
    print(
        "기준일",
        cf.handle.referenceDate(),
        "| 5Y DF 차이",
        cf.handle.discount(5.0) - implied.discount(5.0),
    )
cf.roll(ASOF)
cf.handle.referenceDate()

## 요약

| 객체 | 입력 | 출력 | 언제 쓰나 |
|---|---|---|---|
| `evaluation_date(d)` | 날짜 | with 블록 | QuantLib 계산을 특정 평가일에서 |
| `HELPERS[종류](…)` | 키, Quote, 만기, 구간, Context | `Helper` | `build_curve` 안. 새 시세 종류를 넣을 때 함수 하나 추가 |
| `build_curve(…)` | 레시피, 관례, 시세, asof, linked, 이미 만든 커브, 지수 | `BuiltCurve` | `build_market` 안 |
| `helper_repricing` 등 | BuiltCurve / Handle | DataFrame | 커브 진단 |
| `CurveFactors(base, asof, tenors)` | 날 커브, 격자 | 시장 객체 (`handle`, `factors`, `roll`) | `build_market`이 커브마다 하나 |

## 내 메모

- 